In [3]:
import numpy as np
from sklearn.utils import shuffle

X_speech = np.load("E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_speech.npy")
X_non_1 = np.load("E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_nonspeech_1.npy")
X_non_2 = np.load("E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_nonspeech_2.npy")
X_non = np.concatenate([X_non_1, X_non_2], axis=0)

X = np.vstack([X_non, X_speech]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])
# ===== Shuffle =====
X_train, y_train = shuffle(X, y, random_state=42)
X_test_1 = np.load("E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_speech_aurora.npy")
X_test_2 = np.load("E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/lfcc_lbp_speech_mix_aurora.npy")
X_test = np.concatenate([X_test_1, X_test_2], axis=0)
y_test = np.ones(len(X_test))

print("Final shape:", X_train.shape, y_train.shape, X_test.shape, y_test.shape)

Final shape: (126708, 256) (126708,) (690, 256) (690,)


In [4]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_curve


# ===== Hàm tính EER =====
def compute_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer

speech = np.sum(y == 0)
non_speech = np.sum(y == 1)

scale_pos_weight = speech / non_speech

print("scale_pos_weight:", scale_pos_weight)


model = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


print("===== Result =====")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
#print("EER      :", compute_eer(y_test, y_score))

scale_pos_weight: 0.9792867519565116
===== Result =====
Accuracy : 0.7869565217391304
Precision: 1.0
Recall   : 0.7869565217391304
F1-score : 0.8807785888077859
ROC-AUC  : nan


e:\PythonFile\Project\Voice-Activity-Detect\.venv\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [5]:
from sklearn.ensemble import RandomForestClassifier

speech = np.sum(y == 0)
non_speech = np.sum(y == 1)

scale_pos_weight = speech / non_speech

print("scale_pos_weight:", scale_pos_weight)


model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


print("===== Result =====")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_score))
#print("EER      :", compute_eer(y_test, y_score))

scale_pos_weight: 0.9792867519565116
===== Result =====
Accuracy : 0.6739130434782609
Precision: 1.0
Recall   : 0.6739130434782609
F1-score : 0.8051948051948052
ROC-AUC  : nan


e:\PythonFile\Project\Voice-Activity-Detect\.venv\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
